In [1]:
!pip install transformers datasets sentencepiece evaluate rouge_score
!pip install rouge-score

import pandas as pd
import numpy as np
import torch
import random
import evaluate

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)

from transformers import Seq2SeqTrainingArguments

from transformers import Seq2SeqTrainer

from transformers import set_seed

from tqdm import tqdm

from transformers import T5Tokenizer, T5ForConditionalGeneration

from nltk.translate.bleu_score import corpus_bleu

from rouge_score import rouge_scorer

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

#Read the training set first
train = pd.read_json("/kaggle/input/datasets/zelenezhong/clickbait-spoiler-dataset/train.jsonl", lines=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


In [2]:
#See the first few rows for the training set

train.head()

,uuid,postId,postText,postPlatform,targetParagraphs,targetTitle,targetDescription,targetKeywords,targetMedia,targetUrl,provenance,spoiler,spoilerPositions,tags
0,0af11f6b-c889-4520-9372-66ba25cb7657,532quh,"[Wes Welker Wanted Dinner With Tom Brady, But ...",reddit,[It’ll be just like old times this weekend for...,"Wes Welker Wanted Dinner With Tom Brady, But P...",It'll be just like old times this weekend for ...,"new england patriots, ricky doyle, top stories,","[http://pixel.wp.com/b.gif?v=noscript, http://...",http://nesn.com/2016/09/wes-welker-wanted-dinn...,"{'source': 'anonymized', 'humanSpoiler': 'They...",[how about that morning we go throw?],"[[[3, 151], [3, 186]]]",[passage]
1,b1a1f63d-8853-4a11-89e8-6b2952a393ec,411701128456593408,[NASA sets date for full recovery of ozone hole],Twitter,[2070 is shaping up to be a great year for Mot...,Hole In Ozone Layer Expected To Make Full Reco...,2070 is shaping up to be a great year for Moth...,"ozone layer,ozone hole determined by weather,M...",[http://s.m.huffpost.com/assets/Logo_Huffingto...,http://huff.to/1cH672Z,"{'source': 'anonymized', 'humanSpoiler': '2070...",[2070],"[[[0, 0], [0, 4]]]",[phrase]
2,008b7b19-0445-4e16-8f9e-075b73f80ca4,380537005123190784,[This is what makes employees happy -- and it'...,Twitter,"[Despite common belief, money isn't the key to...",Intellectual Stimulation Trumps Money For Empl...,By: Chad Brooks \r\nPublished: 09/18/2013 06:4...,"employee happiness money,employee happiness in...",[http://i.huffpost.com/gen/1359674/images/o-HA...,http://huff.to/1epfeaw,"{'source': 'anonymized', 'humanSpoiler': 'Inte...",[intellectual stimulation],"[[[1, 186], [1, 210]]]",[phrase]
3,31ecf93c-3e21-4c80-949b-aa549a046b93,844567852531286016,[Passion is overrated — 7 work habits you need...,Twitter,"[It’s common wisdom. Near gospel really, and n...","‘Follow your passion’ is wrong, here are 7 hab...",There's a lot more to work that loving your job,"business, work-life, careers",None,None,"{'source': 'anonymized', 'humanSpoiler': None,...",[Purpose connects us to something bigger and i...,"[[[11, 25], [11, 101]], [[17, 56], [17, 85]], ...",[multi]
4,31b108a3-c828-421a-a4b9-cf651e9ac859,814186311573766144,[The perfect way to cook rice so that it's per...,Twitter,"[Boiling rice may seem simple, but there is a ...",Revealed: The perfect way to cook rice so that...,The question 'How does one cook rice properly?...,"Quora,users,share,perfect,way,cook,rice",None,None,"{'source': 'anonymized', 'humanSpoiler': None,...",[in a rice cooker],"[[[5, 60], [5, 76]]]",[phrase]


In [3]:
#Also, see the column names of the train dataset
train.columns

Index(['uuid', 'postId', 'postText', 'postPlatform', 'targetParagraphs',
       'targetTitle', 'targetDescription', 'targetKeywords', 'targetMedia',
       'targetUrl', 'provenance', 'spoiler', 'spoilerPositions', 'tags'],
      dtype='object')

Based on the dataset description, 14 variables are available. The target variable is spoiler

In [4]:
#Read the test dataset as well as the validation set
test = pd.read_json("/kaggle/input/datasets/zelenezhong/clickbait-spoiler-dataset/test.jsonl", lines=True)

val = pd.read_json("/kaggle/input/datasets/zelenezhong/clickbait-spoiler-dataset/val.jsonl", lines=True)

#See what variables are available in the test dataset
test.columns

Index(['postId', 'postText', 'postPlatform', 'targetParagraphs', 'targetTitle',
       'targetDescription', 'targetKeywords', 'targetMedia', 'targetUrl',
       'id'],
      dtype='object')

For the input variables, we mainly focus on postText, targetTitle, and targetParagraphs. Then we need to change the targetParagraphs from list type to str type.

In [5]:
#Here, we consider to limit the number of paragraphs we use. In this case, if the paragraphs length is smaller than 6, we can use all of them
#If it is longer than 6, then we choose the first four and the last two paragraphs.
def combine_paragraphs(paragraphs):
    if len(paragraphs) <= 6:
        return " ".join(paragraphs)
    return " ".join(paragraphs[:4] + paragraphs[-2:])


train["context"] = train["targetParagraphs"].apply(combine_paragraphs)

val["context"] = val["targetParagraphs"].apply(combine_paragraphs)

test["context"] = test["targetParagraphs"].apply(combine_paragraphs)

Similarly, we need to do the same for postText

In [6]:
def combine_text(x):
    if isinstance(x, list):
        return " ".join(x)
    return x

train["postText"] = train["postText"].apply(combine_text)

val["postText"] = val["postText"].apply(combine_text)

test["postText"] = test["postText"].apply(combine_text)

Then we need to transform the format of the spoiler

In [7]:
def process_spoiler(spoiler):
    return " ".join(spoiler)

train["target_text"] = train["spoiler"].apply(process_spoiler)

val["target_text"] = val["spoiler"].apply(process_spoiler)

So the type of spoiler changes. Next, we create the input

In [8]:
def create_input(row):

    return (
        "generate spoiler: "
        + row["postText"]
        + " title: "
        + row["targetTitle"]
        + " context: "
        + row["context"]
    )


train["input_text"] = train.apply(create_input, axis=1)

val["input_text"] = val.apply(create_input, axis=1)

test["input_text"] = test.apply(create_input, axis=1)

Next, before the formal tokenizer is applied, we need to check and confirm how many pieces of text exceed the maximum length allowed by the model.

In [9]:
#Load the tokenizater first
tokenizer = T5Tokenizer.from_pretrained("t5-base")

#check the length
train_lengths = train["input_text"].apply(
    lambda x: len(tokenizer(x)["input_ids"])
)

print(train_lengths.describe())

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

count     3200.000000
mean       352.372500
std        638.710955
min         27.000000
25%        246.000000
50%        320.000000
75%        404.000000
max      35480.000000
Name: input_text, dtype: float64


In [10]:
(train_lengths > 1024).sum()

np.int64(15)

In [11]:
#We need to check the percentage that exceed max length
(train_lengths > 1024).mean()

np.float64(0.0046875)

This means that 0.468% of the training samples exceed the maximum input length of 1024 tokens that model can handle.

Then, we can create a HuggingFace Dataset.

In [12]:
max_input_length = 1024

train_dataset = Dataset.from_pandas(
    train[["input_text", "target_text"]]
)

val_dataset = Dataset.from_pandas(
    val[["input_text", "target_text"]]
)

test_dataset = Dataset.from_pandas(
    test[["input_text"]]
)

#The generated spoiler is limited to a maximum of 64 tokens.
max_target_length = 64

target_lengths = train["target_text"].apply(
    lambda x: len(tokenizer(x)["input_ids"])
)

target_lengths.describe()

count    3200.000000
mean       21.936563
std        26.996070
min         2.000000
25%         5.000000
50%        12.000000
75%        30.000000
max       337.000000
Name: target_text, dtype: float64

In [13]:
(target_lengths > 64).mean()

np.float64(0.0603125)

Only about 5% of the spoilers exceed the token length of 64, so set it to 64 is reasonable.

Then, we need to load the BART model.

In [14]:
model_name = "t5-base"


model = T5ForConditionalGeneration.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Then, we need to write the tokenizer function

In [15]:
#The max_input_length and max_target_length are both settled before

def tokenize(batch):

    inputs = tokenizer(
        batch["input_text"],
        max_length=max_input_length,
        truncation=True
    )


    targets = tokenizer(
        batch["target_text"],
        max_length=max_target_length,
        truncation=True
    )


    inputs["labels"] = targets["input_ids"]

    return inputs

In [16]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)


val_dataset = val_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Next, we need the Data Collator. During the training process, it combines multiple pieces of data into one batch and performs dynamic processing.

In [17]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

Next, set up TrainingArguments

In [18]:
training_args = Seq2SeqTrainingArguments(

    output_dir="./t5_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=5e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=3,

    weight_decay=0.01,

    predict_with_generate=True,

    load_best_model_at_end=True,

    fp16=True,

    logging_steps=50,

    seed=42,
    data_seed=42
)

Then, create the trainer

In [19]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

Then, apply the training dataset to the model

In [20]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,3.307817,2.975382
2,2.872423,2.948294
3,2.703211,2.950095


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1200, training_loss=2.974285666147868, metrics={'train_runtime': 1118.656, 'train_samples_per_second': 8.582, 'train_steps_per_second': 1.073, 'total_flos': 6538000272261120.0, 'train_loss': 2.974285666147868, 'epoch': 3.0})

In [21]:
#See the evaluate score
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 2.9482944011688232,
 'eval_runtime': 17.5235,
 'eval_samples_per_second': 22.826,
 'eval_steps_per_second': 2.853,
 'epoch': 3.0}

Then, we would like to evaluate how this performs in the validation dataset. We use BLEU, ROUGE-L, and METEOR.

BLEU

In [22]:
predictions = []

model.eval()

for i in tqdm(range(len(val_dataset))):

    inputs = {
        k: torch.tensor(v).unsqueeze(0).to(model.device)
        for k, v in val_dataset[i].items()
        if k in ["input_ids", "attention_mask"]
    }

    with torch.no_grad():

        generated_ids = model.generate(
            **inputs,
            max_length=64,
            num_beams=4
        )

    pred = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    predictions.append(pred)

100%|██████████| 400/400 [05:17<00:00,  1.26it/s]


In [23]:
references = [
    x if isinstance(x, list) else [x]
    for x in val["spoiler"]
]

bleu = evaluate.load("bleu")
bleu_score = bleu.compute(
    predictions=predictions,
    references=references
)
print("BLEU:", bleu_score["bleu"])

BLEU: 0.18019969135040836


ROUGE-L

In [24]:
scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

scores = []

for pred, ref in zip(predictions, references):
    score = scorer.score(ref[0], pred)
    scores.append(score["rougeL"].fmeasure)

rougeL = sum(scores) / len(scores)

print("ROUGE-L:", rougeL)

ROUGE-L: 0.3922146456804335


METEOR

In [25]:
meteor = evaluate.load("meteor")

score = meteor.compute(
    predictions=predictions,
    references=references
)

print(score["meteor"])

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


0.41671120794375954


Take a manual inspection of the spoiler effect generated by T5

In [26]:
for i in range(5):

    sample = val.iloc[i]

    inputs = tokenizer(
        sample["input_text"],
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(model.device)


    generated_ids = model.generate(
        **inputs,
        max_length=64,
        num_beams=4
    )


    pred = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )


    print("="*50)
    print("TRUE:")
    print(sample["target_text"])

    print("PRED:")
    print(pred)

TRUE:
some of the plot elements are so disturbing that they are making him feel sick
PRED:
too dark
TRUE:
"intentionally" could transform a court case against Phoenix-area Sheriff Joe Arpaio from civil charges to a criminal prosecution
PRED:
"intentionally"
TRUE:
20%
PRED:
20%
TRUE:
Alan Rickman & Rupert Grint CBGB
PRED:
CBGB
TRUE:
a man who swallowed a 64GB microSD card and then pooped it into a strainer
PRED:
he couldn't puke it back up, and therefore had to poop it into a pasta strainer and then plug it in to a computer to see if the client’s footage was intact


Then, we use the model on the test dataset

In [27]:
test_inputs = tokenizer(
    test["input_text"].tolist(),
    max_length=1024,
    truncation=True,
    padding=True,
    return_tensors="pt"
)

test_inputs = {
    k: v.to(model.device)
    for k, v in test_inputs.items()
}

In [28]:
#We can also try to change the hyperparameter to improve the prediction.
model.eval()

test_predictions = []

batch_size = 4

for i in tqdm(range(0, len(test), batch_size)):

    batch_texts = test["input_text"].iloc[i:i+batch_size].tolist()

    inputs = tokenizer(
        batch_texts,
        max_length=1024,
        truncation=True,
        padding=True,
        return_tensors="pt"
    ).to(model.device)


    with torch.no_grad():

        generated_ids = model.generate(
            **inputs,
            max_length=64,
            num_beams=4
        )


    decoded = [
        tokenizer.decode(
            ids,
            skip_special_tokens=True
        )
        for ids in generated_ids
    ]

    test_predictions.extend(decoded)


100%|██████████| 100/100 [02:50<00:00,  1.70s/it]


In [29]:
len(test_predictions)

400

In [30]:
for i in range(5):
    print("Prediction", i, ":")
    print(test_predictions[i])
    print()

Prediction 0 :
He has balloons and a sign in hand that reads, "Heard urine need of a kidney, want mine?"

Prediction 1 :
giving at the expense of your own well-being damages your chance of long-term success

Prediction 2 :
higher taxes to fund a local government that develops policies to discourage people from eating crappy food and smoking

Prediction 3 :
Braconid, meaning "any of numerous wasps of the family Braconidae, the larvae of whichare parasitic on aphids and on the larvae of moths, butterflies, beetles."

Prediction 4 :
Cured egg yolks



In [31]:
#Then we have the submission

submission_task2 = pd.DataFrame({
    "id": test["id"],
    "spoiler": test_predictions
})

#Check the head of the submission
submission_task2.head()

,id,spoiler
0,0,"He has balloons and a sign in hand that reads,..."
1,1,giving at the expense of your own well-being d...
2,2,higher taxes to fund a local government that d...
3,3,"Braconid, meaning ""any of numerous wasps of th..."
4,4,Cured egg yolks


In [32]:
submission_task2.to_csv(
    "prediction_task2.csv",
    index=False
)